In [6]:
# -*- coding: utf-8 -*-
"""
COMBINED SENTIMENT AND SARCASM ANALYSIS PIPELINE
This script performs both RoBERTa sentiment analysis and sarcasm detection on hotel reviews.
"""

# =============================================================================
# INSTALLATION AND IMPORTS
# =============================================================================
print("Installing required packages...")
!pip install ollama pandas numpy transformers torch
!pip install emot langdetect num2words contractions

import pandas as pd
import numpy as np
import re
import time
import json
import warnings
warnings.filterwarnings('ignore')

# Import for RoBERTa sentiment analysis
from transformers import pipeline
import torch
from langdetect import detect, DetectorFactory, LangDetectException
from langdetect import detect_langs
from num2words import num2words
import contractions

# Import for sarcasm detection
import ollama
import subprocess
from threading import Thread
from emot.emo_unicode import EMOTICONS_EMO

# Set random seeds for reproducibility
DetectorFactory.seed = 0
torch.manual_seed(42)
np.random.seed(42)

# =============================================================================
# CONFIGURATION - PLEASE REVIEW AND ADJUST THESE SETTINGS
# =============================================================================
# File paths (adjust based on your Google Drive structure)
INPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/hotels.xlsx"
ROBERTA_OUTPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/hotels_roberta_analysis.xlsx"
SARCASM_OUTPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/hotels_sarcasm.xlsx"
COMBINED_OUTPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/hotels_combined_analysis.xlsx"

# Model settings
ROBERTA_MODEL = "cardiffnlp/twitter-roberta-base-sentiment"
SARCASM_MODEL = "deepseek-r1:1.5b"

# Processing settings
DELAY_PER_REVIEW = 0.1  # Delay between sarcasm detection requests
REVIEW_COLUMN = "Review Summary"  # Column containing review text
SAMPLE_SIZE = None  # Set to None for full dataset, or number for testing (e.g., 50)

# =============================================================================
# GOOGLE DRIVE SETUP
# =============================================================================
print("Mounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

# =============================================================================
# SHARED PREPROCESSING FUNCTIONS
# =============================================================================
def handle_emoticons(text):
    """
    Convert emoticons to text descriptions.
    Important for both sentiment and sarcasm analysis.
    Example: :) → "smiling face"
    """
    if not hasattr(handle_emoticons, 'emoticon_patterns'):
        handle_emoticons.emoticon_patterns = []
        for emoticon, description in EMOTICONS_EMO.items():
            escaped_emoticon = re.escape(emoticon)
            handle_emoticons.emoticon_patterns.append(
                (re.compile(escaped_emoticon), f" {description} ")
            )

    text = str(text)
    for pattern, replacement in handle_emoticons.emoticon_patterns:
        text = pattern.sub(replacement, text)
    return text

def is_english_reliable(text):
    """
    Detect if text is English with high confidence.
    Filters out non-English reviews to improve analysis quality.
    """
    if pd.isna(text) or text == "" or len(str(text).strip()) < 10:
        return False

    text = str(text)

    try:
        text_sample = text[:500].strip()
        lang_probabilities = detect_langs(text_sample)

        for lang_prob in lang_probabilities:
            if lang_prob.lang == 'en' and lang_prob.prob >= 0.7:
                return True
            elif lang_prob.lang == 'en' and lang_prob.prob >= 0.5 and len(text_sample) > 50:
                return True

        return False

    except (LangDetectException, Exception):
        text_lower = text.lower()
        english_words = {'the', 'and', 'to', 'of', 'a', 'in', 'that', 'is', 'it', 'for', 'was', 'but', 'with', 'on', 'he', 'she', 'they', 'we', 'you', 'i'}
        words = re.findall(r'\b[a-z]+\b', text_lower)

        if not words or len(words) < 5:
            return False

        english_count = sum(1 for word in words if word in english_words)
        return (english_count / len(words)) > 0.3

# =============================================================================
# RoBERTA SENTIMENT ANALYSIS MODULE
# =============================================================================
def minimal_clean_text_roberta(text):
    """
    Minimal preprocessing optimized for RoBERTa sentiment analysis.
    Preserves meaning while removing noise.
    """
    if pd.isna(text) or text == "":
        return ""

    text = str(text)

    # Expand contractions (e.g., "don't" → "do not")
    try:
        text = contractions.fix(text)
    except:
        pass

    # Handle emoticons
    text = handle_emoticons(text)

    # Remove URLs and emails
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

def run_roberta_analysis(df, review_column=REVIEW_COLUMN):
    """
    Perform sentiment analysis using RoBERTa model.
    Returns dataframe with roberta_score column (1=positive, 0=neutral/negative)
    """
    print("\n" + "="*60)
    print("STARTING RoBERTa SENTIMENT ANALYSIS")
    print("="*60)

    # Create a copy to avoid modifying original data
    df_roberta = df.copy()

    # Preprocess text for RoBERTa
    print("Preprocessing reviews for RoBERTa analysis...")
    df_roberta['text_roberta'] = df_roberta[review_column].apply(minimal_clean_text_roberta)

    # # Filter out very short reviews
    # initial_count = len(df_roberta)
    # df_roberta = df_roberta[df_roberta['text_roberta'].str.split().str.len() > 2]
    # print(f"Removed {initial_count - len(df_roberta)} very short reviews")

    # Initialize RoBERTa pipeline
    device = 0 if torch.cuda.is_available() else -1
    print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

    try:
        sentiment_analyzer = pipeline(
            "sentiment-analysis",
            model=ROBERTA_MODEL,
            tokenizer=ROBERTA_MODEL,
            device=device,
            truncation=True,
            max_length=512,
            batch_size=8
        )
        print("RoBERTa model loaded successfully!")
    except Exception as e:
        print(f"Error loading RoBERTa model: {e}")
        return df

    # Analyze reviews
    print(f"Analyzing {len(df_roberta)} reviews with RoBERTa...")
    reviews_list = df_roberta['text_roberta'].astype(str).tolist()

    results = []
    for i in range(0, len(reviews_list), 100):  # Process in chunks of 100
        chunk = reviews_list[i:i+100]
        try:
            chunk_results = sentiment_analyzer(chunk)
            results.extend(chunk_results)
            if i % 500 == 0:
                print(f"Processed {min(i+100, len(reviews_list))}/{len(reviews_list)} reviews")
        except Exception as e:
            print(f"Error processing chunk: {e}")
            results.extend([{'label': 'LABEL_1', 'score': 0.5}] * len(chunk))

    # Process results (1=positive, 0=neutral/negative)
    def process_roberta_sentiment(result):
        label = result['label']
        return 1 if label == 'LABEL_2' else 0  # LABEL_2 = positive

    df_roberta['roberta_score'] = [process_roberta_sentiment(result) for result in results]
    df_roberta['roberta_confidence'] = [result['score'] for result in results]

    print("RoBERTa analysis completed!")
    positive_count = df_roberta['roberta_score'].sum()
    print(f"Positive reviews: {positive_count}/{len(df_roberta)} ({positive_count/len(df_roberta)*100:.1f}%)")

    return df_roberta

# =============================================================================
# SARCASM DETECTION MODULE
# =============================================================================
def preprocess_for_sarcasm(text):
    """
    Specialized preprocessing for sarcasm detection.
    Preserves case, punctuation, and linguistic cues important for sarcasm.
    """
    if pd.isna(text) or text == "":
        return ""

    text = str(text)

    # Basic cleaning only - preserve case and punctuation!
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\s+', ' ', text)

    # Handle emoticons (crucial for sarcasm)
    text = handle_emoticons(text)

    return text.strip()

class SarcasmDetector:
    """Ollama-based sarcasm detection using deepseek model"""

    def __init__(self, model_name=SARCASM_MODEL):
        self.model = model_name
        self.prompt_template = """
        You are a sarcasm detection expert analyzing hotel reviews. Determine if this review contains sarcasm.
        Respond ONLY in JSON format with these exact keys:
        - "sarcasm": boolean (true/false)
        - "confidence": float between 0.0-1.0
        - "reason": brief explanation
        - "type": either "positive_sarcasm", "negative_sarcasm", or "none"

        Example Response:
        {{"sarcasm": true, "confidence": 0.85, "reason": "Uses positive words ironically", "type": "positive_sarcasm"}}

        Review: {review}
        """

    def analyze_review(self, review):
        """Analyze a single review for sarcasm"""
        try:
            prompt = self.prompt_template.format(review=review)

            for attempt in range(3):  # Retry mechanism
                try:
                    response = ollama.generate(
                        model=self.model,
                        prompt=prompt,
                        options={'temperature': 0.2, 'num_predict': 150}
                    )
                    json_str = response['response'].strip()

                    # Clean JSON response
                    if '```json' in json_str:
                        json_str = json_str.split('```json')[1].split('```')[0].strip()
                    elif '```' in json_str:
                        json_str = json_str.split('```')[1].strip()

                    json_match = re.search(r'\{[\s\S]*?\}', json_str)
                    if json_match:
                        return json.loads(json_match.group(0))

                    break

                except Exception:
                    if attempt == 2:
                        raise
                    time.sleep(2)

            return self.heuristic_fallback(review)

        except Exception as e:
            print(f"Error processing review: {str(e)[:100]}")
            return self.heuristic_fallback(review)

    def heuristic_fallback(self, text):
        """Fallback method when Ollama analysis fails"""
        text_lower = text.lower()
        sarcastic_keywords = ['sarcastic', 'ironic', 'obviously', 'of course', 'surprise', 'wow', 'really']

        is_sarcastic = any(kw in text_lower for kw in sarcastic_keywords)
        confidence = 0.7 if is_sarcastic else 0.3

        return {
            'sarcasm': is_sarcastic,
            'confidence': confidence,
            'reason': 'Heuristic fallback detection',
            'type': 'positive_sarcasm' if is_sarcastic else 'none'
        }

    def process_dataframe(self, df, review_column=REVIEW_COLUMN):
      """Process entire dataframe for sarcasm detection"""
      print(f"Processing {len(df)} reviews for sarcasm...")

      df_sarcasm = df.copy()
      df_sarcasm['sarcasm'] = 0  # 0 = not sarcastic, 1 = sarcastic
      df_sarcasm['sarcasm_confidence'] = 0.0
      df_sarcasm['sarcasm_reason'] = 'Not processed'
      df_sarcasm['sarcasm_type'] = 'none'
      df_sarcasm['processed_review'] = df_sarcasm[review_column].apply(preprocess_for_sarcasm)

      for count, (idx, row) in enumerate(df_sarcasm.iterrows(), 1):
        if count % 10 == 0:
            print(f"Processing review {count}/{len(df_sarcasm)}")

        try:
            # Ensure we have text to process
            if pd.isna(row['processed_review']) or row['processed_review'].strip() == '':
                result = self.heuristic_fallback("Empty review")
            else:
                result = self.analyze_review(row['processed_review'])

            # Convert boolean to 0/1
            df_sarcasm.at[idx, 'sarcasm'] = 1 if result.get('sarcasm', False) else 0
            df_sarcasm.at[idx, 'sarcasm_confidence'] = result.get('confidence', 0.0)
            df_sarcasm.at[idx, 'sarcasm_reason'] = str(result.get('reason', ''))[:200]
            df_sarcasm.at[idx, 'sarcasm_type'] = result.get('type', 'none')

        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            # Apply fallback if any error occurs
            result = self.heuristic_fallback(row['processed_review'])
            df_sarcasm.at[idx, 'sarcasm'] = 1 if result.get('sarcasm', False) else 0
            df_sarcasm.at[idx, 'sarcasm_confidence'] = result.get('confidence', 0.0)
            df_sarcasm.at[idx, 'sarcasm_reason'] = str(result.get('reason', ''))[:200]
            df_sarcasm.at[idx, 'sarcasm_type'] = result.get('type', 'none')

        time.sleep(DELAY_PER_REVIEW)

      sarcastic_count = df_sarcasm['sarcasm'].sum()
      print(f"Sarcasm detection completed! Sarcastic reviews: {sarcastic_count}/{len(df_sarcasm)}")

      return df_sarcasm
# =============================================================================
# MAIN EXECUTION PIPELINE
# =============================================================================
def main_combined_analysis():
    """
    Main function that runs both RoBERTa sentiment analysis and sarcasm detection,
    then combines results into a single output file.
    """
    print("="*80)
    print("COMBINED SENTIMENT AND SARCASM ANALYSIS PIPELINE")
    print("="*80)

    # Step 1: Load data
    print("1. Loading data...")
    try:
        original_df = pd.read_excel(INPUT_FILE)
        print(f"   Loaded {len(original_df)} reviews from {INPUT_FILE}")
    except Exception as e:
        print(f"   Error loading file: {e}")
        return None

    # For testing, use a subset
    if SAMPLE_SIZE and SAMPLE_SIZE < len(original_df):
        print(f"   Using sample of {SAMPLE_SIZE} reviews for testing")
        original_df = original_df.head(SAMPLE_SIZE).copy()

    # Step 2: Filter English reviews (optional but recommended)
    print("2. Filtering English reviews...")
    initial_count = len(original_df)
    english_mask = original_df[REVIEW_COLUMN].apply(is_english_reliable)
    original_df = original_df[english_mask]
    print(f"   Removed {initial_count - len(original_df)} non-English reviews")

    if len(original_df) == 0:
        print("   No English reviews found!")
        return None

    # Step 3: Run RoBERTa sentiment analysis
    print("3. Running RoBERTa sentiment analysis...")
    df_roberta = run_roberta_analysis(original_df)

    # Save intermediate results
    if df_roberta is not None:
        roberta_cols = [col for col in original_df.columns] + ['roberta_score', 'roberta_confidence']
        df_roberta[roberta_cols].to_excel(ROBERTA_OUTPUT_FILE, index=False)
        print(f"   RoBERTa results saved to {ROBERTA_OUTPUT_FILE}")

    # Step 4: Setup Ollama for sarcasm detection
    print("4. Setting up Ollama for sarcasm detection...")
    try:
        !curl -fsSL https://ollama.com/install.sh | sh
        print("   Ollama installed successfully")

        # Start Ollama in background
        def start_ollama():
            subprocess.run(["ollama", "serve"])

        ollama_thread = Thread(target=start_ollama)
        ollama_thread.daemon = True
        ollama_thread.start()
        time.sleep(5)

        !ollama pull {SARCASM_MODEL}
        print("   Model pulled successfully")

    except Exception as e:
        print(f"   Ollama setup failed: {e}")
        print("   Continuing without sarcasm detection...")
        df_sarcasm = original_df.copy()
        df_sarcasm['sarcasm'] = 0
        df_sarcasm['sarcasm_confidence'] = 0.0
        df_sarcasm['sarcasm_reason'] = 'Analysis skipped'
        df_sarcasm['sarcasm_type'] = 'none'

    else:
        # Step 5: Run sarcasm detection
        print("5. Running sarcasm detection...")
        detector = SarcasmDetector()
        df_sarcasm = detector.process_dataframe(original_df)

        # Save intermediate results
        sarcasm_cols = [col for col in original_df.columns] + ['sarcasm', 'sarcasm_confidence', 'sarcasm_reason', 'sarcasm_type']
        df_sarcasm[sarcasm_cols].to_excel(SARCASM_OUTPUT_FILE, index=False)
        print(f"   Sarcasm results saved to {SARCASM_OUTPUT_FILE}")

    # Step 6: Combine results
    print("6. Combining results...")

    # Merge RoBERTa and sarcasm results
    combined_df = original_df.copy()

    # Add RoBERTa scores - FIXED APPROACH
    if 'roberta_score' in df_roberta.columns:
    # Create a mapping between original index and roberta scores
      roberta_scores = df_roberta[['roberta_score', 'roberta_confidence']]

    # Reset indices to ensure proper alignment
      combined_df = combined_df.reset_index(drop=True)
      roberta_scores = roberta_scores.reset_index(drop=True)


      combined_df['roberta_score'] = roberta_scores['roberta_score']
      combined_df['roberta_confidence'] = roberta_scores['roberta_confidence']

# Add sarcasm scores
    sarcasm_cols = ['sarcasm', 'sarcasm_confidence', 'sarcasm_reason', 'sarcasm_type']
    for col in sarcasm_cols:
      if col in df_sarcasm.columns:
          # Reset index for proper alignment
          df_sarcasm = df_sarcasm.reset_index(drop=True)
          combined_df[col] = df_sarcasm[col]

    # Step 7: Save final combined results
    print("7. Saving final results...")
    combined_df.to_excel(COMBINED_OUTPUT_FILE, index=False)
    print(f"   Combined results saved to {COMBINED_OUTPUT_FILE}")

    # Step 8: Display summary
    print("\n" + "="*80)
    print("ANALYSIS SUMMARY")
    print("="*80)
    print(f"Total reviews analyzed: {len(combined_df)}")

    if 'roberta_score' in combined_df.columns:
        positive = combined_df['roberta_score'].sum()
        print(f"Positive sentiment (RoBERTa): {positive} ({positive/len(combined_df)*100:.1f}%)")

    if 'sarcasm' in combined_df.columns:
        sarcastic = combined_df['sarcasm'].sum()
        print(f"Sarcastic reviews detected: {sarcastic} ({sarcastic/len(combined_df)*100:.1f}%)")

    print(f"\nFinal output columns: {list(combined_df.columns)}")
    print("\nSample of final results:")
    sample_cols = ['Hotel name', REVIEW_COLUMN, 'roberta_score', 'sarcasm', 'sarcasm_confidence']
    available_cols = [col for col in sample_cols if col in combined_df.columns]
    print(combined_df[available_cols].head(3))

    return combined_df

# =============================================================================
# EXECUTION
# =============================================================================
if __name__ == "__main__":
    print("Starting combined analysis pipeline...")
    final_results = main_combined_analysis()

    if final_results is not None:
        print("\n✅ Analysis completed successfully!")
        print(f"Final results contain {len(final_results)} reviews")
    else:
        print("\n❌ Analysis failed!")

Installing required packages...
Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Starting combined analysis pipeline...
COMBINED SENTIMENT AND SARCASM ANALYSIS PIPELINE
1. Loading data...
   Loaded 6780 reviews from /content/drive/MyDrive/Colab Notebooks/hotels.xlsx
   Using sample of 100 reviews for testing
2. Filtering English reviews...
   Removed 43 non-English reviews
3. Running RoBERTa sentiment analysis...

STARTING RoBERTa SENTIMENT ANALYSIS
Preprocessing reviews for RoBERTa analysis...
Using device: GPU


Device set to use cuda:0


RoBERTa model loaded successfully!
Analyzing 57 reviews with RoBERTa...
Processed 57/57 reviews
RoBERTa analysis completed!
Positive reviews: 50/57 (87.7%)
   RoBERTa results saved to /content/drive/MyDrive/Colab Notebooks/hotels_roberta_analysis.xlsx
4. Setting up Ollama for sarcasm detection...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
   Ollama installed successfully

   Model pulled successfully
5. Running sarcasm detection...
Processing 57 reviews for sarcasm...
Processing review 10/57
Processing review 20/57
Processing review 30/57
Processing review 40/57
Processing review 50/57
Sarcasm d